# Amazon ML Challenge 2026 — Business Entity Resolution (full pipeline)

Blocking (per-country hashed TF-IDF top-k: name / address / combined lists) → rapidfuzz + context pair features →
LightGBM (MIT) → threshold + one-owner decision tuned for macro F0.5 → `matching_results.tsv` + `candidate_pairs.tsv`.

**Setup on Kaggle**
1. Add the challenge data as a Kaggle Dataset containing `train/` (4 TSVs) and `test/` (3 TSVs) — the cell below finds it automatically under `/kaggle/input`.
2. Settings → **Accelerator: None (CPU)** (the pipeline is CPU-bound; a GPU gives no speedup). **Internet: On** (only to `pip install rapidfuzz unidecode`).
3. Run all. Outputs land in `/kaggle/working/output/`, the submission zip in `/kaggle/working/`.

No external data, no pretrained models; all normalization dictionaries are hand-written in `normalize.py`.

In [ ]:
!pip install -q rapidfuzz unidecode
import os, glob, sys, platform
print(platform.python_version(), "cpus:", os.cpu_count())
!free -g

In [ ]:
# locate the dataset directory (the folder that contains train/ and test/)
hits = glob.glob("/kaggle/input/**/train_source1.tsv", recursive=True)
assert hits, "train_source1.tsv not found under /kaggle/input - attach the dataset"
DATA_DIR = os.path.dirname(os.path.dirname(hits[0]))
print("DATA_DIR =", DATA_DIR)
for sp in ("train", "test"):
    print(sp, sorted(os.listdir(f"{DATA_DIR}/{sp}")))
os.makedirs("/kaggle/working/src", exist_ok=True)
os.makedirs("/kaggle/working/utils", exist_ok=True)
%cd /kaggle/working

## Source modules (written to `/kaggle/working/src/`)

In [ ]:
%%writefile src/config.py
import os
SEED = 42

# train/val subsample (fraction of train S1 per country); val is a disjoint slice
TRAIN_FRAC = 0.10
VAL_FRAC = 0.04
MIN_PARTITION = 50  # country labels with fewer records fall back to the whole pool

# blocking
S1_CHUNK = 2000
POOL_CHUNK = 1_000_000
NAME_MAX_DF = 2000   # absolute doc freq; more frequent tokens are not used for retrieval
ADDR_MAX_DF = 2000
K_NAME = 20   # retrieval depth per list (wide net)
K_ADDR = 30
K_COMB = 40
A_NAME = 5    # per-list rank caps used for the final candidate set
A_ADDR = 10
A_COMB = 15
MAX_CAND = 20       # b3 rank-cap selection (kept for reference)
STAGE1_RECALL = 0.995   # fraction of wide-net true pairs the learned pruner must keep (val)
STAGE1_MAX_CAND = 40
TOPK_JOBS = max(2, (os.cpu_count() or 4) - 1)
BLOCK_VERSION = "b4"  # bump when blocking/normalization changes (cache key)
FEAT_VERSION = "f3"

# model
LGB_PARAMS = dict(objective="binary", learning_rate=0.08, num_leaves=63, min_child_samples=50,
                  feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, seed=SEED,
                  verbose=-1, num_threads=os.cpu_count())
NUM_ROUNDS = 1500
EARLY_STOP = 50

# decision (tuned on validation; overwritten by tuning output in work/decision.json when present)
THR = 0.5
REL = 0.0

In [ ]:
%%writefile src/io_utils.py
import os
import pyarrow as pa
import pyarrow.csv as pacsv


def read_tsv(path, columns=None):
    """Read a challenge TSV as all-string pandas DataFrame (no quoting, no NA inference)."""
    with open(path, encoding="utf-8") as f:
        header = f.readline().rstrip("\n").split("\t")
    tbl = pacsv.read_csv(
        path,
        parse_options=pacsv.ParseOptions(delimiter="\t", quote_char=False, newlines_in_values=False),
        convert_options=pacsv.ConvertOptions(
            column_types={c: pa.string() for c in header},
            strings_can_be_null=False,
            quoted_strings_can_be_null=False,
            null_values=[],
            include_columns=columns,
        ),
        read_options=pacsv.ReadOptions(block_size=1 << 24),
    )
    df = tbl.to_pandas(types_mapper=lambda t: None)
    return df


def write_id_lists(path, s1_ids, lists, col):
    """Write source1_entity_id<TAB><col> with comma-joined ids; lists aligned to s1_ids."""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write(f"source1_entity_id\t{col}\n")
        for s1, ids in zip(s1_ids, lists):
            if ids:
                seen = dict.fromkeys(ids)
                f.write(f"{s1}\t{','.join(seen)}\n")
            else:
                f.write(f"{s1}\t\n")

In [ ]:
%%writefile src/evaluate.py
import os
import sys
import time
import numpy as np
import pandas as pd


def rss_mb():
    try:
        if sys.platform == "win32":
            import ctypes
            from ctypes import wintypes

            class PMC(ctypes.Structure):
                _fields_ = [("cb", wintypes.DWORD), ("PageFaultCount", wintypes.DWORD),
                            ("PeakWorkingSetSize", ctypes.c_size_t), ("WorkingSetSize", ctypes.c_size_t),
                            ("QuotaPeakPagedPoolUsage", ctypes.c_size_t), ("QuotaPagedPoolUsage", ctypes.c_size_t),
                            ("QuotaPeakNonPagedPoolUsage", ctypes.c_size_t), ("QuotaNonPagedPoolUsage", ctypes.c_size_t),
                            ("PagefileUsage", ctypes.c_size_t), ("PeakPagefileUsage", ctypes.c_size_t)]
            pmc = PMC()
            pmc.cb = ctypes.sizeof(PMC)
            k32, psapi = ctypes.WinDLL("kernel32"), ctypes.WinDLL("psapi")
            k32.GetCurrentProcess.restype = ctypes.c_void_p
            psapi.GetProcessMemoryInfo.argtypes = [ctypes.c_void_p, ctypes.POINTER(PMC), wintypes.DWORD]
            psapi.GetProcessMemoryInfo(k32.GetCurrentProcess(), ctypes.byref(pmc), pmc.cb)
            return pmc.WorkingSetSize / 2**20, pmc.PeakWorkingSetSize / 2**20
        import resource
        r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
        return r, r
    except Exception:
        return float("nan"), float("nan")


class Stage:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t = time.time()
        return self

    def __exit__(self, *a):
        cur, peak = rss_mb()
        print(f"[stage] {self.name}: {time.time()-self.t:.1f}s rss={cur:.0f}MB peak={peak:.0f}MB", flush=True)


def f05(pred, true):
    if not true and not pred:
        return 1.0
    if not true or not pred:
        return 0.0
    tp = len(pred & true)
    if tp == 0:
        return 0.0
    p, r = tp / len(pred), tp / len(true)
    return 1.25 * p * r / (0.25 * p + r)


def score(pred_map, true_map, s1_ids, country=None, verbose=True):
    """pred_map/true_map: dict s1 -> set. Returns dict of metrics."""
    fs, tp, npred, ntrue = [], 0, 0, 0
    sing_ok, sing_n = 0, 0
    for s in s1_ids:
        p, t = pred_map.get(s, set()), true_map.get(s, set())
        fs.append(f05(p, t))
        tp += len(p & t); npred += len(p); ntrue += len(t)
        if not t:
            sing_n += 1; sing_ok += (not p)
    fs = np.array(fs)
    out = dict(F05=fs.mean(), P=tp / max(npred, 1), R=tp / max(ntrue, 1), singleton_acc=sing_ok / max(sing_n, 1), n=len(fs))
    if country is not None:
        c = pd.Series(country, index=range(len(fs)))
        for k, idx in c.groupby(c).groups.items():
            out[f"F05_{k}"] = fs[list(idx)].mean()
        nt = np.array([min(len(true_map.get(s, ())), 3) for s in s1_ids])
        for b in range(4):
            if (nt == b).any():
                out[f"F05_m{b}{'+' if b == 3 else ''}"] = fs[nt == b].mean()
    if verbose:
        print("  ".join(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}" for k, v in out.items()), flush=True)
    return out


def sanity():
    assert f05({"a", "b"}, {"a", "b"}) == 1.0
    assert abs(f05({"a", "b", "c"}, {"a", "c"}) - 0.7142857) < 1e-6
    assert f05(set(), set()) == 1.0 and f05({"a"}, set()) == 0.0
    tm = {"x": set(), "y": {"a"}, "z": set(), "w": {"b"}}
    assert score({}, tm, list(tm), verbose=False)["F05"] == 0.5
    assert score(tm, tm, list(tm), verbose=False)["F05"] == 1.0
    print("scorer sanity OK (perfect=1.0, empty=singleton frac, example=0.714)")


def blocking_report(cand, gt_pairs, s1_ids, true_map):
    """cand: DataFrame(s1, cid). gt_pairs: DataFrame(source1_entity_id, mid)."""
    s1set = set(s1_ids)
    g = gt_pairs[gt_pairs["source1_entity_id"].isin(s1set)]
    key_c = set(zip(cand["s1"], cand["cid"]))
    hit = np.fromiter(((a, b) in key_c for a, b in zip(g["source1_entity_id"], g["mid"])), bool, len(g))
    per = cand.groupby("s1").size().reindex(s1_ids, fill_value=0)
    cmap = cand.groupby("s1")["cid"].agg(set).to_dict()
    ceil = np.mean([f05(true_map.get(s, set()) & cmap.get(s, set()), true_map.get(s, set())) for s in s1_ids])
    out = dict(pair_recall=hit.mean(), ceiling_F05=ceil, mean_cands=per.mean(), median_cands=per.median(),
               p95_cands=per.quantile(0.95), max_cands=per.max(), total_pairs=len(cand))
    print("blocking: " + "  ".join(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}" for k, v in out.items()), flush=True)
    return out


if __name__ == "__main__":
    sanity()

In [ ]:
%%writefile src/normalize.py
"""Country-agnostic text normalization. All dictionaries are hand-written constants."""
import re
import os
import time
from multiprocessing import Pool

import pandas as pd
from unidecode import unidecode

LEGAL = {
    "inc": "inc", "incorporated": "inc", "corp": "corp", "corporation": "corp", "co": "co", "company": "co",
    "llc": "llc", "llp": "llp", "lp": "lp", "ltd": "ltd", "limited": "ltd", "pvt": "pvt", "private": "pvt",
    "plc": "plc", "pc": "pc", "pllc": "pllc", "sarl": "sarl", "sas": "sas", "sasu": "sasu", "sa": "sa",
    "eurl": "eurl", "sci": "sci", "snc": "snc", "gmbh": "gmbh", "tbk": "tbk", "lllp": "llp", "ag": "ag",
    "bv": "bv", "nv": "nv", "pty": "pty", "opc": "opc",
}
NAME_STOP = {"the", "and", "of", "et", "de", "du", "des", "la", "le", "les", "l", "d"}
WEB_RE = re.compile(r"^(?:https?://)?(?:www\.)?([a-z0-9\-]+)\.(?:com|net|org|in|co\.in|fr|biz|info|us|co)$")
LEET = str.maketrans({"0": "o", "1": "l", "3": "e", "4": "a", "5": "s", "7": "t", "@": "a", "$": "s"})
NONALNUM = re.compile(r"[^a-z0-9 ]+")
MIXED = re.compile(r"(?=.*[a-z])(?=.*\d)")
SPACES = re.compile(r"\s+")

ADDR_ABBR = {
    "st": "street", "str": "street", "rd": "road", "ave": "avenue", "av": "avenue", "blvd": "boulevard",
    "bd": "boulevard", "bld": "boulevard", "dr": "drive", "ln": "lane", "hwy": "highway", "pkwy": "parkway",
    "cv": "cove", "ct": "court", "cir": "circle", "pl": "place", "sq": "square", "ter": "terrace",
    "trl": "trail", "tr": "trail", "pk": "park", "mt": "mount", "ft": "fort", "twp": "township",
    "ste": "suite", "fl": "floor", "flr": "floor", "apt": "apartment", "bldg": "building", "nr": "near",
    "opp": "opposite", "r": "rue", "rte": "route", "ch": "chemin", "imp": "impasse", "all": "allee",
    "fbg": "faubourg", "n": "north", "s": "south", "e": "east", "w": "west", "ne": "northeast",
    "nw": "northwest", "se": "southeast", "sw": "southwest", "mg": "marg", "extn": "extension",
    "ext": "extension", "sec": "sector", "ph": "phase", "vill": "village", "po": "post", "dist": "district",
    "cplx": "complex", "hsg": "housing", "soc": "society", "stn": "station", "jn": "junction",
}
ADDR_DROP = {"null", "none", "na", "no", "number", "cdp", "pmb", "unit", "suite", "apartment", "floor",
             "de", "du", "des", "la", "le", "les", "l", "d", "the", "of", "and", "cedex"}
STATES = {  # code -> name, applied to address tokens (US + India codes, hand-written)
    "al": "alabama", "ak": "alaska", "az": "arizona", "ar": "arkansas", "ca": "california", "co": "colorado",
    "ct": "connecticut", "de": "delaware", "fl": "florida", "ga": "georgia", "hi": "hawaii", "id": "idaho",
    "il": "illinois", "in": "indiana", "ia": "iowa", "ks": "kansas", "ky": "kentucky", "la": "louisiana",
    "me": "maine", "md": "maryland", "ma": "massachusetts", "mi": "michigan", "mn": "minnesota",
    "ms": "mississippi", "mo": "missouri", "mt": "montana", "ne": "nebraska", "nv": "nevada",
    "nh": "newhampshire", "nj": "newjersey", "nm": "newmexico", "ny": "newyork", "nc": "northcarolina",
    "nd": "northdakota", "oh": "ohio", "ok": "oklahoma", "or": "oregon", "pa": "pennsylvania",
    "ri": "rhodeisland", "sc": "southcarolina", "sd": "southdakota", "tn": "tennessee", "tx": "texas",
    "ut": "utah", "vt": "vermont", "va": "virginia", "wa": "washington", "wv": "westvirginia",
    "wi": "wisconsin", "wy": "wyoming", "dc": "districtofcolumbia",
}
STATE_NAMES = {  # multiword state names -> single token
    "new hampshire": "newhampshire", "new jersey": "newjersey", "new mexico": "newmexico", "new york": "newyork",
    "north carolina": "northcarolina", "north dakota": "northdakota", "rhode island": "rhodeisland",
    "south carolina": "southcarolina", "south dakota": "southdakota", "west virginia": "westvirginia",
    "district of columbia": "districtofcolumbia",
    "uttar pradesh": "uttarpradesh", "madhya pradesh": "madhyapradesh", "andhra pradesh": "andhrapradesh",
    "himachal pradesh": "himachalpradesh", "arunachal pradesh": "arunachalpradesh", "tamil nadu": "tamilnadu",
    "west bengal": "westbengal", "jammu and kashmir": "jammukashmir", "jammu kashmir": "jammukashmir",
}
IN_STATES = {"up": "uttarpradesh", "mp": "madhyapradesh", "ap": "andhrapradesh", "hp": "himachalpradesh",
             "tn": "tamilnadu", "wb": "westbengal", "mh": "maharashtra", "rj": "rajasthan", "gj": "gujarat",
             "ka": "karnataka", "kl": "kerala", "dl": "delhi", "hr": "haryana", "pb": "punjab", "br": "bihar",
             "tg": "telangana", "ts": "telangana", "or": "odisha", "od": "odisha", "jh": "jharkhand",
             "uk": "uttarakhand", "cg": "chhattisgarh", "jk": "jammukashmir"}
# Devanagari state names transliterated by unidecode (hand-written aliases)
TRANSLIT = {"uttr prdesh": "uttarpradesh", "raajsthaan": "rajasthan", "mhaaraassttr": "maharashtra",
            "dillii": "delhi", "hriyaannaa": "haryana", "gujraat": "gujarat", "krnaattk": "karnataka",
            "bihaar": "bihar", "mdhy prdesh": "madhyapradesh", "tmilnaaddu": "tamilnadu",
            "pshcim bngaal": "westbengal", "telngaanaa": "telangana", "pnjaab": "punjab", "kerl": "kerala"}
STATE_CODE = {v: k for k, v in STATES.items()}
STATE_CODE.update({v: k for k, v in IN_STATES.items() if k not in ("ts", "od", "or")})
LANDMARK = {"near", "opposite", "behind", "next", "beside", "adjacent", "above", "below"}
NUM_RE = re.compile(r"\d+")


SKEL_SUBS = [("ph", "f"), ("sh", "s"), ("kh", "k"), ("gh", "g"), ("th", "t"), ("dh", "d"), ("bh", "b"),
             ("ch", "c"), ("ck", "k"), ("q", "k"), ("w", "v"), ("z", "j"), ("x", "ks"), ("c", "k")]
VOWELS = re.compile(r"(?<!^)[aeiouyh]")
REPEAT = re.compile(r"(.)\1+")
TRANSLIT_LEGAL = {"prvt", "prbt", "prvr", "prvtt", "lmtd", "lmtt", "lmrd", "lmtdd", "lmd"}


def skeleton(t):
    for a, b in SKEL_SUBS:
        t = t.replace(a, b)
    return REPEAT.sub(r"\1", VOWELS.sub("", t))


def _ascii(s):
    return s if s.isascii() else unidecode(s)


def norm_name(raw):
    s = _ascii(raw).lower().replace("�", "").replace("&", " and ")
    toks = []
    for t in s.split():
        m = WEB_RE.match(t.strip(".,;:()[]"))
        toks.append(m.group(1).replace("-", "") if m else t)
    s = NONALNUM.sub(" ", " ".join(toks))
    out = []
    for t in s.split():
        if MIXED.match(t) and sum(c.isalpha() for c in t) >= 2:
            t = t.translate(LEET)
        out.append(t)
    norm = " ".join(out)
    legal = sorted({LEGAL[t] for t in out if t in LEGAL})
    core = [t for t in out if t not in LEGAL and t not in NAME_STOP and skeleton(t) not in TRANSLIT_LEGAL]
    if not core:
        core = out
    return norm, " ".join(core), " ".join(legal)


def norm_addr(raw):
    s = _ascii(raw).lower().replace("�", "")
    s = NONALNUM.sub(" ", s.replace("#", " "))
    s = SPACES.sub(" ", s).strip()
    for k, v in STATE_NAMES.items():
        if k in s:
            s = s.replace(k, v)
    for k, v in TRANSLIT.items():
        if k in s:
            s = re.sub(r"\b" + k + r"\b", v, s)
    toks, nums, main = [], [], []
    skip = 0
    for t in s.split():
        if t.isdigit():
            t = t.lstrip("0") or "0"
            nums.append(t)
        else:
            if t in ADDR_DROP:
                continue
            t = ADDR_ABBR.get(t, t)
            t = STATE_CODE.get(t, t)
        toks.append(t)
        if t in LANDMARK:
            skip = 3
        elif skip:
            skip -= 1
        else:
            main.append(t)
    postcode = ""
    for n in nums:
        if len(n) in (5, 6):
            postcode = n
            break
    return " ".join(toks), " ".join(dict.fromkeys(nums)), postcode, " ".join(main)


def _norm_chunk(args):
    names, addrs = args
    n = [norm_name(x) for x in names]
    a = [norm_addr(x) for x in addrs]
    return n, a


def normalize_df(df, n_jobs=os.cpu_count(), chunk=100_000):
    """Return DataFrame with normalized fields for a raw source DataFrame."""
    names = df["business_name"].tolist()
    addrs = df["business_address"].tolist()
    jobs = [(names[i:i + chunk], addrs[i:i + chunk]) for i in range(0, len(names), chunk)]
    del names, addrs
    res_n, res_a = [], []
    with Pool(n_jobs) as p:
        for n, a in p.imap(_norm_chunk, jobs):
            res_n.extend(n)
            res_a.extend(a)
    del jobs
    out = pd.DataFrame({"entity_id": df["entity_id"].values, "country": df["country"].values})
    out["name_norm"], out["name_core"], out["name_legal"] = zip(*res_n) if res_n else ([], [], [])
    del res_n
    out["addr_norm"], out["addr_nums"], out["postcode"], out["addr_main"] = zip(*res_a) if res_a else ([], [], [], [])
    return out


def normalize_split(work, split, force=False):
    """Normalize raw parquet for split in {'train','test'} -> work/norm_<split>_s{1,2,3}.parquet"""
    for i in (1, 2, 3):
        dst = f"{work}/norm_{split}_s{i}.parquet"
        if os.path.exists(dst) and not force:
            continue
        t = time.time()
        df = pd.read_parquet(f"{work}/raw_{split}_s{i}.parquet")
        out = normalize_df(df)
        out.to_parquet(dst, index=False)
        print(f"normalized {split}_s{i} rows={len(out)} {time.time()-t:.0f}s", flush=True)
        del df, out


if __name__ == "__main__":
    import sys
    ex = ["Meek Pr�ncipa1 Co", "johnsviking.com", "THE INTERSTATE BIOPHARMA INC.", "Family Charities-II Corporation",
          "Quiroz [Yhn]", "राम मार्केटिंग प्राइवेट लिमिटेड", "Supérieure Sante EURL", "Maa Security Private Ltd",
          "LLC Global Compeelfmte Vendome", "Tayl0r Battery Holdings"]
    for e in ex:
        print(e, "->", norm_name(e))
    ad = ["##2620 SAWMILL RD, FORT RECOVERY, OH", "02620 Sawmill Rd, Recovery Twp, Ohio",
          "KH NO-436/437, KHUSHAL PARK MAIN ROAD, GHAZIABAD, उत्तर प्रदेश", "राजस्थान, P NO 4, CAFCUTTA FARM HOUSE, JANPATH, JAIPUR, NULL",
          "3100-3104 SEMINOLE LOOP, CROSSVILLE, TN", "12 RUE de lOrne, 44800, Saint-Herblain, Pays de la Loire",
          "51 R DE LA PLANCHE AU GUE, NANTES, Pays de la Loire", "Shop 4, Near SBI ATM, MG Rd, Pune 411001",
          "56 BELKNAP ST, NULL, WESTOROUGH CDP, MA", "Pessac, Gironde, 60 AVENUE DU PARC D'ESPAGNE"]
    for e in ad:
        print(e, "->", norm_addr(e))

In [ ]:
%%writefile src/blocking.py
"""Candidate generation, per country partition.
Hashed TF-IDF spaces: name (words, 4-char prefixes/suffixes, consonant skeletons, concatenation, bigrams) and
address (word uni+bigrams). Three top-k lists per S1: name, address, combined (name+address). Top-k is computed as
S1-chunk x pool-chunk sparse products merged with argpartition; features with doc freq > MAX_DF are excluded from the
retrieval product (keeps it sparse => sub-quadratic). Final candidates = top-a name ∪ top-b addr ∪ top-c comb,
capped at MAX_CAND per S1 by best list rank."""
import gc
from multiprocessing import Pool

import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import HashingVectorizer

import config as C
from evaluate import Stage
from normalize import skeleton

NF = 2 ** 22


def name_tokens(s):
    toks = s.split()
    out = list(toks) + ["~" + skeleton(t) for t in toks if len(t) >= 3]
    for t in toks:
        if len(t) >= 6:
            out.append("<" + t[:4]); out.append(">" + t[-4:])
    if toks:
        out.append("#" + "".join(toks))
    for a, b in zip(toks, toks[1:]):
        out.append(a + "_" + b)
    return out


def addr_tokens(s):
    toks = s.split()
    return toks + [a + "_" + b for a, b in zip(toks, toks[1:])]


def _hv(kind):
    return HashingVectorizer(analyzer=name_tokens if kind == "name" else addr_tokens, n_features=NF,
                             alternate_sign=False, norm=None, dtype=np.float32)


def _hash_job(args):
    kind, texts = args
    m = _hv(kind).transform(texts)
    m.data = np.log1p(m.data, dtype=np.float32)  # sublinear tf
    return m


def hash_texts(kind, texts, chunk=200_000, n_jobs=os.cpu_count()):
    jobs = [(kind, texts[i:i + chunk]) for i in range(0, len(texts), chunk)]
    if not jobs:
        return sp.csr_matrix((0, NF), dtype=np.float32)
    if len(jobs) == 1:
        return _hash_job(jobs[0])
    with Pool(n_jobs) as p:
        mats = p.map(_hash_job, jobs)
    return sp.vstack(mats, format="csr")


class Space:
    """Hashed TF-IDF space fitted on a partition (S1 + pool texts)."""

    def __init__(self, max_df):
        self.max_df = max_df

    def fit(self, mats):
        n = sum(m.shape[0] for m in mats)
        df = np.zeros(NF, np.int64)
        for m in mats:
            df += np.bincount(m.indices, minlength=NF)
        self.idf = (np.log((1 + n) / (1 + df)) + 1).astype(np.float32)
        self.keep = (df <= self.max_df).astype(np.float32)
        return self

    def weight(self, m):
        m = m.copy()
        m.data *= self.idf[m.indices]
        norms = np.sqrt(np.asarray(m.multiply(m).sum(1)).ravel()).astype(np.float32)
        norms[norms == 0] = 1
        return sp.diags(1 / norms).dot(m).tocsr().astype(np.float32)


def _retrieval(m, keep):
    r = m.copy()
    r.data *= keep[r.indices]
    r.eliminate_zeros()
    return r


_PT = None


def _init_worker(PT):
    global _PT
    _PT = PT


def _chunk_topk(args):
    """Top-k of one S1 chunk against the worker's pool chunk. Returns local (idx, score) arrays."""
    Qc, k, p0 = args
    R = (Qc @ _PT).tocsr()
    n = R.shape[0]
    bi = np.full((n, k), -1, np.int32)
    bs = np.full((n, k), -1, np.float32)
    ip, ix, dt = R.indptr, R.indices, R.data
    for r in range(n):
        a, b = ip[r], ip[r + 1]
        if a == b:
            continue
        d, c = dt[a:b], ix[a:b]
        if b - a > k:
            sel = np.argpartition(-d, k)[:k]
            d, c = d[sel], c[sel]
        bs[r, :len(d)] = d
        bi[r, :len(d)] = c + p0
    return bi, bs


def _merge(best_i, best_s, rows, ci, cs, k):
    s_all = np.concatenate([best_s[rows], cs], axis=1)
    i_all = np.concatenate([best_i[rows], ci], axis=1)
    top = np.argpartition(-s_all, k - 1, axis=1)[:, :k]
    best_s[rows] = np.take_along_axis(s_all, top, 1)
    best_i[rows] = np.take_along_axis(i_all, top, 1)


def topk_search(Q, P, k, s1_chunk=C.S1_CHUNK, pool_chunk=C.POOL_CHUNK, n_jobs=C.TOPK_JOBS):
    """Q: (nq,F) csr, P: (np,F) csr. Returns idx (nq,k) int32 (-1 = none) and score (nq,k) float32.
    Loops over pool chunks; within a pool chunk, S1 chunks are scored in parallel worker processes."""
    nq = Q.shape[0]
    best_i = np.full((nq, k), -1, np.int32)
    best_s = np.full((nq, k), -1, np.float32)
    starts = list(range(0, nq, s1_chunk))
    for p0 in range(0, P.shape[0], pool_chunk):
        PT = P[p0:p0 + pool_chunk].T.tocsr()
        jobs = [(Q[q0:q0 + s1_chunk], k, p0) for q0 in starts]
        if n_jobs > 1 and len(jobs) > 1:
            with Pool(n_jobs, initializer=_init_worker, initargs=(PT,)) as pl:
                for q0, (ci, cs) in zip(starts, pl.imap(_chunk_topk, jobs)):
                    _merge(best_i, best_s, slice(q0, q0 + len(ci)), ci, cs, k)
        else:
            _init_worker(PT)
            for q0, job in zip(starts, jobs):
                ci, cs = _chunk_topk(job)
                _merge(best_i, best_s, slice(q0, q0 + len(ci)), ci, cs, k)
            _init_worker(None)
        del PT, jobs
        gc.collect()
    return best_i, best_s


def _ranked(idx, score, q0=0):
    """Flatten a top-k result to (q, p, rank) with rank 1 = best; drops empty slots. q offset by q0."""
    order = np.argsort(-score, axis=1, kind="stable")
    idx = np.take_along_axis(idx, order, 1)
    nq, k = idx.shape
    q = np.repeat(np.arange(q0, q0 + nq, dtype=np.int32), k)
    rank = np.tile(np.arange(1, k + 1, dtype=np.int16), nq)
    p = idx.ravel()
    m = p >= 0
    return q[m], p[m], rank[m]


def rowwise_cos(A, B):
    return np.asarray(A.multiply(B).sum(1)).ravel().astype(np.float32)


STAGE1_FEATS = ["r_name", "r_addr", "r_comb", "name_cos", "addr_cos", "comb", "s1_rank_comb", "s1_rank_name",
                "s1_gap_comb", "s1_gap_name", "s1_gap_addr", "s1_n_union", "n_lists"]


def stage1_features(df):
    """Cheap per-pair blocking features; all per-S1 aggregates, so chunking by S1 is exact."""
    g = df.groupby("q")
    df["comb"] = df["name_cos"] + df["addr_cos"]
    df["s1_rank_comb"] = g["comb"].rank(ascending=False, method="first").astype(np.float32)
    df["s1_rank_name"] = g["name_cos"].rank(ascending=False, method="first").astype(np.float32)
    df["s1_gap_comb"] = (g["comb"].transform("max") - df["comb"]).astype(np.float32)
    df["s1_gap_name"] = (g["name_cos"].transform("max") - df["name_cos"]).astype(np.float32)
    df["s1_gap_addr"] = (g["addr_cos"].transform("max") - df["addr_cos"]).astype(np.float32)
    df["s1_n_union"] = g["p"].transform("size").astype(np.float32)
    df["n_lists"] = ((df["r_name"] < 99).astype(np.int8) + (df["r_addr"] < 99) + (df["r_comb"] < 99)).astype(np.float32)
    return df


def prune(df, pruner):
    """Apply the learned stage-1 blocker: keep p1 >= thr, always keep each S1's top-1 by comb and by name_cos,
    cap at MAX_CAND by p1. Output = final candidate set scored by the matcher."""
    bst, thr, max_cand = pruner
    df["p1"] = bst.predict(df[STAGE1_FEATS].values.astype(np.float32), num_threads=C.TOPK_JOBS)
    keep = (df["p1"].values >= thr) | (df["s1_rank_comb"].values == 1) | (df["s1_rank_name"].values == 1)
    d = df[keep].sort_values(["q", "p1"], ascending=[True, False])
    d = d[d.groupby("q").cumcount() < max_cand]
    return d.sort_values(["q", "p"]).reset_index(drop=True)


def block_partition(s1, pool, k_name=C.K_NAME, k_addr=C.K_ADDR, k_comb=C.K_COMB, pruner=None, q_chunk=100_000):
    """s1/pool: normalized DataFrames (positional). Returns the union of the three top-k lists with ranks (99 = not
    in that list), cosines and stage-1 features; pruned by `pruner` when given. Union/pruning run in S1 chunks."""
    with Stage(f"hash n_s1={len(s1)} n_pool={len(pool)}"):
        qn, pn = hash_texts("name", s1["name_core"].tolist()), hash_texts("name", pool["name_core"].tolist())
        qa, pa = hash_texts("addr", s1["addr_norm"].tolist()), hash_texts("addr", pool["addr_norm"].tolist())
        sn, sa = Space(C.NAME_MAX_DF).fit([qn, pn]), Space(C.ADDR_MAX_DF).fit([qa, pa])
        Qn, Pn, Qa, Pa = sn.weight(qn), sn.weight(pn), sa.weight(qa), sa.weight(pa)
        del qn, pn, qa, pa
        gc.collect()
    with Stage("topk name"):
        ln = topk_search(_retrieval(Qn, sn.keep), _retrieval(Pn, sn.keep), k_name)
    with Stage("topk addr"):
        la = topk_search(_retrieval(Qa, sa.keep), _retrieval(Pa, sa.keep), k_addr)
    with Stage("topk combined"):
        keep = np.concatenate([sn.keep, sa.keep])
        Qc = _retrieval(sp.hstack([Qn, Qa], format="csr"), keep)
        Pc = _retrieval(sp.hstack([Pn, Pa], format="csr"), keep)
        lc = topk_search(Qc, Pc, k_comb)
        del Qc, Pc
        gc.collect()
    out = []
    with Stage("union + cosines + stage1" + (" + prune" if pruner else "")):
        for c0 in range(0, len(s1), q_chunk):
            c1 = c0 + q_chunk
            parts = []
            for col, (ii, ss) in (("r_name", ln), ("r_addr", la), ("r_comb", lc)):
                q, p, r = _ranked(ii[c0:c1], ss[c0:c1], c0)
                d = pd.DataFrame({"q": q, "p": p, "r_name": np.int16(99), "r_addr": np.int16(99), "r_comb": np.int16(99)})
                d[col] = r
                parts.append(d)
            df = pd.concat(parts, ignore_index=True).groupby(["q", "p"], sort=False).min().reset_index()
            del parts
            name_cos = np.empty(len(df), np.float32); addr_cos = np.empty(len(df), np.float32)
            step = 1_000_000
            for i in range(0, len(df), step):
                qq, pp = df["q"].values[i:i + step], df["p"].values[i:i + step]
                name_cos[i:i + step] = rowwise_cos(Qn[qq], Pn[pp])
                addr_cos[i:i + step] = rowwise_cos(Qa[qq], Pa[pp])
            df["name_cos"], df["addr_cos"] = name_cos, addr_cos
            df = stage1_features(df)
            if pruner:
                df = prune(df, pruner)
            out.append(df)
            gc.collect()
    return pd.concat(out, ignore_index=True)


def select(df, a=C.A_NAME, b=C.A_ADDR, c=C.A_COMB, max_cand=C.MAX_CAND):
    """Per-list rank cap then per-S1 cap by best rank (ties broken by name_cos+addr_cos)."""
    d = df[(df["r_name"] <= a) | (df["r_addr"] <= b) | (df["r_comb"] <= c)]
    best = np.minimum(np.minimum(d["r_name"].values, d["r_addr"].values), d["r_comb"].values)
    d = d.assign(_b=best, _s=-(d["name_cos"].values + d["addr_cos"].values)).sort_values(["q", "_b", "_s"])
    d = d[d.groupby("q").cumcount() < max_cand]
    return d.drop(columns=["_b", "_s"]).sort_values(["q", "p"]).reset_index(drop=True)


def partitions(s1_country, pool_country):
    """Yield (label, s1_mask, pool_mask); labels with too few records fall back to the whole pool."""
    s1_country = pd.Series(s1_country); pool_country = pd.Series(pool_country)
    pc = pool_country.value_counts()
    for lab, n in s1_country.value_counts().items():
        s1m = (s1_country == lab).values
        if n >= C.MIN_PARTITION and pc.get(lab, 0) >= C.MIN_PARTITION:
            yield lab, s1m, (pool_country == lab).values
        else:
            yield lab, s1m, np.ones(len(pool_country), bool)

In [ ]:
%%writefile src/features.py
"""Pair features. String similarities via rapidfuzz in a multiprocessing pool over aligned pair chunks."""
from multiprocessing import Pool

import os
import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler, Levenshtein

from normalize import skeleton

FUZZ_COLS = ["n_ratio", "n_partial", "n_tsort", "n_tset", "n_jw", "n_nospace", "nn_tset", "n_first_eq", "n_acr",
             "n_tok_jacc", "a_tset", "a_partial", "a_tsort", "am_tset", "a_tok_jacc", "num_jacc", "num_first_eq",
             "num_conflict", "num_both", "pc_eq", "pc_conflict", "legal_eq", "legal_conflict", "legal_one",
             "n_len_a", "n_len_b", "n_ntok_b", "a_len_b", "num_near", "n_skel", "n_contain"]


def _jacc(a, b):
    if not a and not b:
        return -1.0
    return len(a & b) / len(a | b)


def _num_near(u, v):
    for x in u:
        for y in v:
            if x == y or (len(x) >= 2 and len(y) >= 2 and (x.startswith(y) or y.startswith(x) or x.endswith(y)
                                                          or y.endswith(x))) or Levenshtein.distance(x, y) <= 1:
                return 1.0
    return 0.0


def _skel(s):
    return " ".join(skeleton(t) for t in s.split())


def _pair_chunk(args):
    na, nb, nna, nnb, aa, ab, ama, amb, numa, numb, pca, pcb, la, lb = args
    out = np.zeros((len(na), len(FUZZ_COLS)), np.float32)
    for i in range(len(na)):
        a, b = na[i], nb[i]
        ta, tb = a.split(), b.split()
        sa, sb = set(ta), set(tb)
        acr_a = "".join(t[0] for t in ta) if len(ta) > 1 else ""
        acr_b = "".join(t[0] for t in tb) if len(tb) > 1 else ""
        xa, xb = aa[i], ab[i]
        stxa, stxb = set(xa.split()), set(xb.split())
        nua, nub = numa[i].split(), numb[i].split()
        su, sv = set(nua), set(nub)
        both = bool(su) and bool(sv)
        pa, pb = pca[i], pcb[i]
        ga, gb = set(la[i].split()), set(lb[i].split())
        out[i] = (
            fuzz.ratio(a, b), fuzz.partial_ratio(a, b), fuzz.token_sort_ratio(a, b), fuzz.token_set_ratio(a, b),
            JaroWinkler.normalized_similarity(a, b), fuzz.ratio(a.replace(" ", ""), b.replace(" ", "")),
            fuzz.token_set_ratio(nna[i], nnb[i]),
            float(bool(ta) and bool(tb) and ta[0] == tb[0]),
            float(bool(acr_a and acr_a == b.replace(" ", "")) or bool(acr_b and acr_b == a.replace(" ", ""))),
            _jacc(sa, sb),
            fuzz.token_set_ratio(xa, xb) if xa and xb else -1, fuzz.partial_ratio(xa, xb) if xa and xb else -1,
            fuzz.token_sort_ratio(xa, xb) if xa and xb else -1,
            fuzz.token_set_ratio(ama[i], amb[i]) if ama[i] and amb[i] else -1,
            _jacc(stxa, stxb),
            _jacc(su, sv) if both else -1, float(both and nua[0] == nub[0]), float(both and not (su & sv)), float(both),
            float(bool(pa) and pa == pb), float(bool(pa) and bool(pb) and pa != pb),
            float(bool(ga) and ga == gb), float(bool(ga) and bool(gb) and not (ga & gb)), float(bool(ga) != bool(gb)),
            len(a), len(b), len(tb), len(xb),
            _num_near(su, sv) if both else -1, fuzz.token_set_ratio(_skel(a), _skel(b)),
            float(bool(sa) and bool(sb) and (sa <= sb or sb <= sa)),
        )
    return out


def string_features(s1, pool, q, p, n_jobs=os.cpu_count(), chunk=100_000):
    """q, p: global positional index arrays into s1/pool normalized frames."""
    cols = {}
    for c in ["name_core", "name_norm", "addr_norm", "addr_main", "addr_nums", "postcode", "name_legal"]:
        cols[c] = (s1[c].values, pool[c].values)
    jobs = []
    for i in range(0, len(q), chunk):
        qq, pp = q[i:i + chunk], p[i:i + chunk]
        args = []
        for c in ["name_core", "name_norm", "addr_norm", "addr_main", "addr_nums", "postcode", "name_legal"]:
            A, B = cols[c]
            args += [A[qq].tolist(), B[pp].tolist()]
        # reorder to match _pair_chunk signature
        na, nb, nna, nnb, aa, ab, ama, amb, numa, numb, pca, pcb, la, lb = args
        jobs.append((na, nb, nna, nnb, aa, ab, ama, amb, numa, numb, pca, pcb, la, lb))
    res = []
    with Pool(n_jobs) as pl:
        for r in pl.imap(_pair_chunk, jobs):
            res.append(r)
    del jobs
    arr = np.vstack(res) if res else np.zeros((0, len(FUZZ_COLS)), np.float32)
    return pd.DataFrame(arr, columns=FUZZ_COLS)


def context_features(df):
    """df has q, p, name_cos, addr_cos, comb (all S1 candidates of the processed set)."""
    out = pd.DataFrame(index=df.index)
    g = df.groupby("q")
    out["rank_comb"] = g["comb"].rank(ascending=False, method="min").astype(np.float32)
    out["rank_name"] = g["name_cos"].rank(ascending=False, method="min").astype(np.float32)
    out["rank_addr"] = g["addr_cos"].rank(ascending=False, method="min").astype(np.float32)
    out["gap_comb"] = (g["comb"].transform("max") - df["comb"]).astype(np.float32)
    out["gap_name"] = (g["name_cos"].transform("max") - df["name_cos"]).astype(np.float32)
    out["gap_addr"] = (g["addr_cos"].transform("max") - df["addr_cos"]).astype(np.float32)
    out["n_cands"] = g["p"].transform("size").astype(np.float32)
    return out

In [ ]:
%%writefile src/model.py
import json
import lightgbm as lgb
import numpy as np

import config as C

BASE_FEATS = ["name_cos", "addr_cos", "comb", "from_name", "from_addr", "from_comb", "src_s3"]


def train(X, y, groups, feats, work):
    """Train LightGBM with early stopping on a held-out 10% of S1 groups."""
    rng = np.random.RandomState(C.SEED)
    ug = np.unique(groups)
    hold = set(rng.choice(ug, size=max(1, len(ug) // 10), replace=False))
    m = np.fromiter((g in hold for g in groups), bool, len(groups))
    dtr = lgb.Dataset(X[~m], y[~m], feature_name=feats, free_raw_data=True)
    dva = lgb.Dataset(X[m], y[m], feature_name=feats, reference=dtr)
    bst = lgb.train(C.LGB_PARAMS, dtr, C.NUM_ROUNDS, valid_sets=[dva],
                    callbacks=[lgb.early_stopping(C.EARLY_STOP, verbose=False), lgb.log_evaluation(200)])
    bst.save_model(f"{work}/lgb.txt")
    json.dump(feats, open(f"{work}/features.json", "w"))
    imp = sorted(zip(feats, bst.feature_importance("gain")), key=lambda x: -x[1])
    print("best_iter", bst.best_iteration, "top gain:", [(f, int(g)) for f, g in imp[:15]], flush=True)
    return bst


def load(work):
    return lgb.Booster(model_file=f"{work}/lgb.txt"), json.load(open(f"{work}/features.json"))

In [ ]:
%%writefile src/decide.py
import numpy as np
import pandas as pd

from evaluate import f05


def decide(df, thr, rel, one_owner=True):
    """df: q, p, prob. Returns filtered DataFrame of accepted (q, p)."""
    d = df[df["prob"] >= thr]
    if one_owner and len(d):
        d = d[d["prob"] >= d.groupby("p")["prob"].transform("max")]
        d = d[~d.duplicated("p")]
    if rel > 0 and len(d):
        d = d[d["prob"] >= rel * d.groupby("q")["prob"].transform("max")]
    return d


def fast_macro_f05(acc, true_cnt, q_all):
    """acc: accepted DataFrame with q and label (1 = true pair). true_cnt: Series q -> #true matches (0 for singletons)."""
    g = acc.groupby("q")["label"].agg(["sum", "size"])
    tc = true_cnt.reindex(q_all).fillna(0).values
    tp = g["sum"].reindex(q_all).fillna(0).values
    npred = g["size"].reindex(q_all).fillna(0).values
    f = np.zeros(len(q_all))
    f[(tc == 0) & (npred == 0)] = 1.0
    m = (tp > 0)
    P = np.where(m, tp / np.maximum(npred, 1), 0); R = np.where(m, tp / np.maximum(tc, 1), 0)
    f[m] = 1.25 * P[m] * R[m] / (0.25 * P[m] + R[m])
    return f.mean(), tp.sum() / max(npred.sum(), 1), tp.sum() / max(tc.sum(), 1)


def tune(df, true_cnt, q_all, one_owner=True):
    best = (-1, None)
    rows = []
    for thr in np.arange(0.20, 0.94, 0.04):
        for rel in [0.0, 0.5, 0.7]:
            acc = decide(df, thr, rel, one_owner)
            f, P, R = fast_macro_f05(acc, true_cnt, q_all)
            rows.append((round(thr, 2), rel, f, P, R))
            if f > best[0]:
                best = (f, (round(float(thr), 2), rel, P, R))
    return best, pd.DataFrame(rows, columns=["thr", "rel", "F05", "P", "R"])

In [ ]:
%%writefile src/run_pipeline.py
"""End-to-end pipeline: raw TSV -> normalization -> per-country blocking -> pair features -> LightGBM -> decision -> TSVs.
  python src/run_pipeline.py --data-dir <dataset> --out-dir <output> --work-dir <work> --mode {val,full}
"""
import argparse
import gc
import json
import os
import sys

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config as C
from evaluate import Stage, score, blocking_report, sanity
from io_utils import read_tsv, write_id_lists

FEAT_COLS = ["name_core", "name_norm", "name_legal", "addr_norm", "addr_main", "addr_nums", "postcode"]


def load_raw(data_dir, work, split, force=False):
    for i in (1, 2, 3):
        dst = f"{work}/raw_{split}_s{i}.parquet"
        if force or not os.path.exists(dst):
            read_tsv(f"{data_dir}/{split}/{split}_source{i}.tsv").to_parquet(dst, index=False)


class Source:
    """Lazy column access to normalized parquet (S1, or pool = S2 then S3) via Arrow; rows fetched by int index."""

    def __init__(self, work, split, which):
        self.paths = [f"{work}/norm_{split}_s1.parquet"] if which == "s1" else \
            [f"{work}/norm_{split}_s2.parquet", f"{work}/norm_{split}_s3.parquet"]

    def col(self, c):
        import pyarrow as pa
        return pa.concat_arrays([pq.read_table(p, columns=[c]).column(c).combine_chunks() for p in self.paths])

    def frame(self, cols, idx=None):
        import pyarrow as pa
        out = {}
        for c in cols:
            a = self.col(c)
            if idx is not None:
                a = a.take(pa.array(idx))
            out[c] = a.to_numpy(zero_copy_only=False)
            del a
        return pd.DataFrame(out)


def load_gt(data_dir, work):
    p = f"{work}/gt_pairs.parquet"
    if not os.path.exists(p):
        gt = read_tsv(f"{data_dir}/train/train_ground_truth.tsv")
        gt = gt.assign(mid=gt["matched_entity_ids"].str.split(",")).explode("mid")
        gt = gt[gt["mid"] != ""][["source1_entity_id", "mid"]]
        gt.to_parquet(p, index=False)
    return pd.read_parquet(p)


def make_split(s1_ids, s1_country, gt):
    """Positional (train_idx, val_idx): per country x has-match stratified subsample of train S1 (seed 42)."""
    has = pd.Series(s1_ids).isin(set(gt["source1_entity_id"])).values
    strata = pd.Series(s1_country).astype(str).values + "_" + has.astype(str)
    rng = np.random.RandomState(C.SEED)
    tr, va = [], []
    for _, idx in pd.Series(np.arange(len(s1_ids))).groupby(strata):
        idx = rng.permutation(idx.values)
        ntr, nva = int(round(len(idx) * C.TRAIN_FRAC)), int(round(len(idx) * C.VAL_FRAC))
        tr.append(idx[:ntr]); va.append(idx[ntr:ntr + nva])
    return np.sort(np.concatenate(tr)), np.sort(np.concatenate(va))


def safe(lab):
    return "".join(ch if ch.isalnum() else "_" for ch in str(lab))


def wide_country(work, split, lab, q_glob, p_glob, S1, POOL, pruner=None, force=False):
    """Wide-net blocking for one country (pruned inline when pruner is given). Global int32 q/p."""
    from blocking import block_partition
    path = f"{work}/{'cands' if pruner else 'wide'}_{split}_{safe(lab)}_{C.BLOCK_VERSION}_{len(q_glob)}.parquet"
    if os.path.exists(path) and not force:
        return pd.read_parquet(path)
    with Stage(f"blocking {split} {lab} n_s1={len(q_glob)} n_pool={len(p_glob)} pruned={pruner is not None}"):
        s1t = S1.frame(["name_core", "addr_norm"], q_glob)
        pt = POOL.frame(["name_core", "addr_norm"], p_glob)
        df = block_partition(s1t, pt, pruner=pruner)
        del s1t, pt
        df["q"] = q_glob[df["q"].values].astype(np.int32)
        df["p"] = p_glob[df["p"].values].astype(np.int32)
        df.to_parquet(path, index=False)
    gc.collect()
    return df


def process_country(work, split, lab, q_glob, p_glob, S1, POOL, force=False, predictor=None, pruner=None,
                    chunk_s1=150_000):
    """Block + features for one country, features computed in S1 chunks. Returns DataFrame with global int32 q/p and
    feature columns, or (when predictor=(booster, feats) is given) only q, p, prob."""
    from features import string_features, context_features
    tag = f"{split}_{safe(lab)}_{C.BLOCK_VERSION}_{len(q_glob)}"
    cpath = f"{work}/cands_{tag}.parquet"
    fpath = f"{work}/{'preds' if predictor else 'feats'}_{tag}_{C.FEAT_VERSION}.parquet"
    if os.path.exists(fpath) and not force:
        return pd.read_parquet(fpath)
    if os.path.exists(cpath) and not force:
        cand = pd.read_parquet(cpath)
    else:
        cand = wide_country(work, split, lab, q_glob, p_glob, S1, POOL, pruner=pruner, force=force)
    with Stage(f"name/addr frequency {split} {lab}"):
        pnc = POOL.frame(["name_core", "addr_norm"], p_glob)
        name_freq = pnc["name_core"].value_counts()
        addr_freq = pnc["addr_norm"].value_counts()
        del pnc
    uq_all = np.unique(cand["q"].values)
    outs = []
    for c0 in range(0, len(uq_all), chunk_s1):
        with Stage(f"features {split} {lab} chunk {c0}"):
            part = cand[cand["q"].isin(uq_all[c0:c0 + chunk_s1])].reset_index(drop=True)
            uq, qi = np.unique(part["q"].values, return_inverse=True)
            up, pi = np.unique(part["p"].values, return_inverse=True)
            s1f = S1.frame(FEAT_COLS, uq)
            pf = POOL.frame(FEAT_COLS + ["entity_id"], up)
            part["src_s3"] = pd.Series(pf["entity_id"].values[pi]).str.startswith("S3-").values.astype(np.int8)
            sf = string_features(s1f, pf, qi.astype(np.int64), pi.astype(np.int64))
            cf = context_features(part)
            cf["p_name_freq"] = pf["name_core"].map(name_freq).fillna(0).values[pi].astype(np.float32)
            cf["q_name_freq"] = s1f["name_core"].map(name_freq).fillna(0).values[qi].astype(np.float32)
            pa_ = pf["addr_norm"].values[pi]
            cf["p_addr_freq"] = np.where(pa_ == "", -1, pf["addr_norm"].map(addr_freq).fillna(0).values[pi]).astype(np.float32)
            F = pd.concat([part, sf, cf.reset_index(drop=True)], axis=1)
            del s1f, pf, sf, cf, part
            if predictor:
                bst, feats = predictor
                F = pd.DataFrame({"q": F["q"].values, "p": F["p"].values,
                                  "prob": bst.predict(F[feats].values.astype(np.float32), num_threads=os.cpu_count())})
            outs.append(F)
            gc.collect()
    F = pd.concat(outs, ignore_index=True) if outs else pd.DataFrame(columns=["q", "p", "prob"])
    F.to_parquet(fpath, index=False)
    return F


def run_split(work, split, s1_idx, force=False, predictor=None, pruner=None):
    S1, POOL = Source(work, split, "s1"), Source(work, split, "pool")
    s1c = S1.col("country").to_numpy(zero_copy_only=False)
    pc = POOL.col("country").to_numpy(zero_copy_only=False)
    from blocking import partitions
    parts = []
    for lab, s1m, poolm in partitions(s1c[s1_idx], pc):
        F = process_country(work, split, lab, s1_idx[s1m].astype(np.int32), np.flatnonzero(poolm).astype(np.int32),
                            S1, POOL, force, predictor, pruner)
        F["country_part"] = lab
        parts.append(F)
        gc.collect()
    return pd.concat(parts, ignore_index=True), S1, POOL


def model_feats(F):
    drop = {"q", "p", "country_part", "label", "prob", "p1"}
    return [c for c in F.columns if c not in drop]


def ids_for(src, idx):
    import pyarrow as pa
    return src.col("entity_id").take(pa.array(idx)).to_numpy(zero_copy_only=False)


def gt_int(work, S1, POOL, gt):
    """Ground-truth pairs as global int indices (q into S1, p into pool), via Arrow hash lookups."""
    import pyarrow as pa
    import pyarrow.compute as pc
    path = f"{work}/gt_int.parquet"
    if not os.path.exists(path):
        q = pc.index_in(pa.array(gt["source1_entity_id"].values), value_set=S1.col("entity_id")).to_numpy(zero_copy_only=False)
        p = pc.index_in(pa.array(gt["mid"].values), value_set=POOL.col("entity_id")).to_numpy(zero_copy_only=False)
        pd.DataFrame({"q": q.astype(np.int64), "p": p.astype(np.int64)}).to_parquet(path, index=False)
    return pd.read_parquet(path)


def int_labels(df, gi):
    key = df["q"].values.astype(np.int64) << 32 | df["p"].values.astype(np.int64)
    gkey = gi["q"].values << 32 | gi["p"].values
    return np.isin(key, gkey).astype(np.int8)


def train_stage1(Wd, lab, is_va, n_true_va, work):
    """Learned first-stage blocker on cheap features; threshold keeps >= STAGE1_RECALL of wide-net val positives."""
    import lightgbm as lgb
    from blocking import STAGE1_FEATS, prune
    prm = dict(objective="binary", learning_rate=0.1, num_leaves=31, min_child_samples=100, seed=C.SEED, verbose=-1,
               num_threads=os.cpu_count())
    bst = lgb.train(prm, lgb.Dataset(Wd.loc[~is_va, STAGE1_FEATS].values.astype(np.float32), lab[~is_va]), 100)
    bst.save_model(f"{work}/stage1.txt")
    V = Wd.loc[is_va].copy()
    V["label"] = lab[is_va]
    wide_pos = V["label"].sum()
    print(f"stage1: val wide-net pair_recall={wide_pos / n_true_va:.4f} mean_cands={len(V) / V['q'].nunique():.2f}", flush=True)
    best = None
    for thr in [0.5, 0.3, 0.2, 0.1, 0.07, 0.05, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.002, 0.001]:
        d = prune(V.copy(), (bst, thr, C.STAGE1_MAX_CAND))
        kept = d["label"].sum() / wide_pos
        n = d.groupby("q").size()
        print(f"  thr={thr}: kept {kept:.4f} of wide positives, pair_recall={d['label'].sum() / n_true_va:.4f} "
              f"mean={n.mean():.2f} p95={n.quantile(.95):.0f}", flush=True)
        if kept >= C.STAGE1_RECALL:
            best = thr
            break
    best = best if best is not None else 0.001
    json.dump({"thr": best}, open(f"{work}/stage1.json", "w"))
    return (bst, best, C.STAGE1_MAX_CAND)


def add_labels(F, S1, POOL, gt):
    s1_ids = ids_for(S1, F["q"].values)
    p_ids = ids_for(POOL, F["p"].values)
    g = gt[gt["source1_entity_id"].isin(set(s1_ids))]
    key = set(zip(g["source1_entity_id"], g["mid"]))
    F["label"] = np.fromiter(((a, b) in key for a, b in zip(s1_ids, p_ids)), np.int8, len(F))
    return g


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", required=True)
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--work-dir", required=True)
    ap.add_argument("--mode", choices=["val", "full"], required=True)
    ap.add_argument("--force", action="store_true")
    a = ap.parse_args()
    W = a.work_dir
    os.makedirs(W, exist_ok=True); os.makedirs(a.out_dir, exist_ok=True)
    sanity()
    from normalize import normalize_split
    from model import train as lgb_train
    from decide import decide, tune
    with Stage("load+normalize"):
        for sp in ["train"] + (["test"] if a.mode == "full" else []):
            load_raw(a.data_dir, W, sp)
            normalize_split(W, sp)
    gt = load_gt(a.data_dir, W)
    S1 = Source(W, "train", "s1")
    s1_ids = S1.col("entity_id").to_numpy(zero_copy_only=False)
    s1_country = S1.col("country").to_numpy(zero_copy_only=False)
    tr_idx, va_idx = make_split(s1_ids, s1_country, gt)
    print(f"subsample: train S1={len(tr_idx)} val S1={len(va_idx)} of {len(s1_ids)} (per-country stratified, seed {C.SEED})", flush=True)
    va_ids, va_country = s1_ids[va_idx], s1_country[va_idx]
    true_map = gt[gt["source1_entity_id"].isin(set(va_ids))].groupby("source1_entity_id")["mid"].agg(set).to_dict()
    del s1_ids, s1_country
    gc.collect()

    sub = np.sort(np.concatenate([tr_idx, va_idx]))
    POOL = Source(W, "train", "pool")
    gi = gt_int(W, S1, POOL, gt)
    va_arr = np.asarray(va_idx)
    n_true_va = int(np.isin(gi["q"].values, va_arr).sum())
    from blocking import partitions, prune
    s1c = S1.col("country").to_numpy(zero_copy_only=False)
    pcn = POOL.col("country").to_numpy(zero_copy_only=False)
    parts = [(lab, sub[s1m].astype(np.int32), np.flatnonzero(poolm).astype(np.int32))
             for lab, s1m, poolm in partitions(s1c[sub], pcn)]
    del s1c, pcn
    if os.path.exists(f"{W}/stage1.json") and os.path.exists(f"{W}/stage1.txt") and not a.force:
        import lightgbm as lgb
        pruner = (lgb.Booster(model_file=f"{W}/stage1.txt"), json.load(open(f"{W}/stage1.json"))["thr"], C.STAGE1_MAX_CAND)
    else:
        Wd = pd.concat([wide_country(W, "train", lab, q, p, S1, POOL, force=a.force) for lab, q, p in parts],
                       ignore_index=True)
        with Stage("stage1 blocker train + threshold"):
            wl = int_labels(Wd, gi)
            pruner = train_stage1(Wd, wl, np.isin(Wd["q"].values, va_arr), n_true_va, W)
        del Wd, wl
        gc.collect()
    print(f"stage1 pruner thr={pruner[1]} max_cand={pruner[2]}", flush=True)
    for lab, q, p in parts:
        cpath = f"{W}/cands_train_{safe(lab)}_{C.BLOCK_VERSION}_{len(q)}.parquet"
        if not os.path.exists(cpath) or a.force:
            with Stage(f"prune train {lab}"):
                wd = wide_country(W, "train", lab, q, p, S1, POOL)
                prune(wd, pruner).to_parquet(cpath, index=False)
                del wd
                gc.collect()
    F, S1, POOL = run_split(W, "train", sub, a.force)
    F["label"] = int_labels(F, gi)
    feats = model_feats(F)
    is_va = F["q"].isin(set(va_idx)).values

    # blocking metrics on validation S1
    Fv = F.loc[is_va, ["q", "p", "label"]]
    blocking_report(pd.DataFrame({"s1": ids_for(S1, Fv["q"].values), "cid": ids_for(POOL, Fv["p"].values)}),
                    gt, va_ids, true_map)

    with Stage("lgb train (train split)"):
        bst = lgb_train(F.loc[~is_va, feats].values.astype(np.float32), F.loc[~is_va, "label"].values,
                        F.loc[~is_va, "q"].values, feats, W)
    dv = F.loc[is_va, ["q", "p", "label"]].copy()
    dv["prob"] = bst.predict(F.loc[is_va, feats].values.astype(np.float32), num_threads=os.cpu_count())
    tc = pd.Series({q: len(true_map.get(s, ())) for q, s in zip(va_idx, va_ids)})
    with Stage("tune decision"):
        (f, (thr, rel, P, R)), grid = tune(dv, tc, va_idx)
        f_no = tune(dv, tc, va_idx, one_owner=False)[0]
    print(f"VAL BEST F05={f:.4f} thr={thr} rel={rel} P={P:.4f} R={R:.4f} | without one-owner F05={f_no[0]:.4f} {f_no[1]}", flush=True)
    json.dump({"thr": thr, "rel": rel, "val_F05": f, "best_iter": bst.best_iteration}, open(f"{W}/decision.json", "w"))
    grid.to_csv(f"{W}/decision_grid.csv", index=False)
    acc = decide(dv, thr, rel)
    pm = pd.DataFrame({"s": ids_for(S1, acc["q"].values), "c": ids_for(POOL, acc["p"].values)}).groupby("s")["c"].agg(set).to_dict()
    score(pm, true_map, list(va_ids), country=va_country)
    dv.to_parquet(f"{W}/val_preds.parquet", index=False)
    if a.mode == "val":
        return

    # full: predict test with the model validated above (trained on the train split of the subsample)
    del F, dv
    gc.collect()
    TS1 = Source(W, "test", "s1")
    n_test = len(TS1.col("entity_id"))
    Ft, TS1, TPOOL = run_split(W, "test", np.arange(n_test), a.force, predictor=(bst, feats), pruner=pruner)
    with Stage("decide test"):
        acc = decide(Ft[["q", "p", "prob"]], thr, rel)
    write_outputs(TS1, TPOOL, Ft[["q", "p"]], acc, a.out_dir, n_test)


def write_outputs(TS1, TPOOL, cand, acc, out_dir, n_test):
    ids1 = TS1.col("entity_id").to_numpy(zero_copy_only=False)
    country = TS1.col("country").to_numpy(zero_copy_only=False)
    cp = ids_for(TPOOL, cand["p"].values); ap_ = ids_for(TPOOL, acc["p"].values)
    cl = pd.Series(cp).groupby(cand["q"].values).agg(list)
    ml = pd.Series(ap_).groupby(acc["q"].values).agg(list)
    cl_all = [cl.get(i, []) for i in range(n_test)]
    ml_all = [ml.get(i, []) for i in range(n_test)]
    write_id_lists(f"{out_dir}/candidate_pairs.tsv", ids1, cl_all, "candidate_entity_ids")
    write_id_lists(f"{out_dir}/matching_results.tsv", ids1, ml_all, "matched_entity_ids")
    n = np.array([len(x) for x in cl_all]); m = np.array([len(x) for x in ml_all])
    print(f"test: S1={n_test} cands mean={n.mean():.2f} median={np.median(n)} p95={np.percentile(n, 95)} max={n.max()} "
          f"| matched mean={m.mean():.2f} empty_frac={(m == 0).mean():.4f}", flush=True)
    for c in pd.unique(country):
        mk = country == c
        print(f"  {c}: n={mk.sum()} mean_cands={n[mk].mean():.2f} mean_matches={m[mk].mean():.2f} empty={(m[mk] == 0).mean():.4f}", flush=True)


if __name__ == "__main__":
    main()

## Submission validator
Copy `utils/validate_submission.py` from the official challenge kit into `/kaggle/working/utils/` (it is not redistributed here). The next cell runs it if present.


## Run
`--mode full`: normalize → split train S1 (10% train / 4% val per country × has-match, seed 42) → block against the full
per-country pool → features → LightGBM → tune (threshold, relative rule) on validation → print val metrics → block + score
test → write outputs. Intermediate caches in `work/` (delete it to recompute from scratch).

In [ ]:
!mkdir -p work output
!PYTHONUNBUFFERED=1 python src/run_pipeline.py --data-dir "{DATA_DIR}" --out-dir output --work-dir work --mode full 2>&1 | grep -v "^\[LightGBM\]"

In [ ]:
import os
if os.path.exists('utils/validate_submission.py'):
    !python utils/validate_submission.py --matching output/matching_results.tsv --candidate output/candidate_pairs.tsv --test-dir "{DATA_DIR}/test"
else:
    print('validator not found: copy it from the challenge kit')
!head -3 output/matching_results.tsv; wc -l output/*.tsv


In [ ]:
# optional: package outputs + code (add README.md / requirements.txt / Documentation_template.md to the zip as needed)
!mkdir -p pkg/output pkg/code/business_entity_resolution/src
!cp output/*.tsv pkg/output/ && cp src/*.py pkg/code/business_entity_resolution/src/
!cd pkg && zip -qr ../team_submission.zip . && cd .. && unzip -l team_submission.zip